In [ ]:
# Apply Custom CSS Styling (Section 1)
from IPython.display import display, HTML
import ipywidgets as widgets

display(HTML("""
<!-- MathJax config -->
<script type="text/x-mathjax-config">
MathJax.Hub.Config({
  tex2jax: {
    inlineMath: [['$','$'], ['\\\\(','\\\\)']],
    displayMath: [['$$','$$'], ['\\\\[','\\\\]']],
    processEscapes: true,
    processEnvironments: true,
    skipTags: ['script', 'noscript', 'style', 'textarea', 'pre']
  },
  TeX: {
    equationNumbers: { autoNumber: "AMS" },
    extensions: ["AMSmath.js", "AMSsymbols.js"]
  }
});
</script>
<script src="https://cdn.jsdelivr.net/npm/mathjax@2/MathJax.js?config=TeX-AMS_HTML"></script>

<style>
    body {
        font-family: 'Helvetica Neue', Arial, sans-serif;
        line-height: 1.5;
        max-width: 100%;
        margin: 0 auto;
        padding: 20px;
    }
    h1 {
        color: #2c3e50;
        text-align: center;
        padding-bottom: 15px;
        border-bottom: 2px solid #3498db;
        margin-bottom: 30px;
    }
    h2 {
        color: #3498db;
        margin-top: 30px;
        padding-bottom: 10px;
        border-bottom: 1px solid #eee;
    }
    h3 {
        color: #2980b9;
        margin-top: 25px;
    }
    .section {
        margin: 30px 0;
        padding: 20px;
        background: #f8f9fa;
        border-radius: 5px;
        border-left: 5px solid #3498db;
    }
    .author-info {
        display: block;
        width: 100%;
        background: #f8f9fa;
        padding: 20px;
        border-radius: 8px;
        margin: 30px auto;
        border-left: 5px solid #3498db;
        box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        clear: both;
        box-sizing: border-box;
    }
    .funding-info {
        display: block;
        width: 100%;
        background: #f0f7fa;
        padding: 20px;
        border-radius: 8px;
        margin: 30px auto;
        border-left: 5px solid #2980b9;
        font-size: 0.95em;
        box-shadow: 0 2px 5px rgba(0,0,0,0.1);
        clear: both;
        box-sizing: border-box;
    }
    .theory {
        background: #f8f9fa;
        padding: 15px;
        border-radius: 5px;
        margin: 20px 0;
        border-left: 5px solid #3498db;
    }
    .widget-area {
        background: #f1f8ff;
        padding: 20px;
        border-radius: 5px;
        margin: 30px 0;
        border: 1px solid #d1e5f9;
    }
    .footer {
        margin-top: 50px;
        padding-top: 20px;
        border-top: 1px solid #eee;
        text-align: left;
        font-size: 0.9em;
        color: #7f8c8d;
    }
    .jupyter-widgets-output-area {
        width: 100% !important;
        max-width: 100% !important;
        overflow-x: auto !important;
    }
    .jupyter-matplotlib-figure {
        width: 100% !important;
        max-width: 100% !important;
    }
    .jupyter-matplotlib-canvas-container {
        width: 100% !important;
    }
    canvas.jupyter-matplotlib-canvas {
        max-width: 100% !important;
        height: auto !important;
    }
    .widget-output {
        width: 100% !important;
        overflow-x: hidden !important;
    }
    .figure {
        max-width: 100% !important;
        margin: 0 auto !important;
    }
</style>
"""))

In [ ]:
# --- Title and Author/Funding Info ---
display(HTML("""
<h1>HTSegregation Pyiron Workflow Demonstrator</h1>
<div class="author-info">
    <h2>Authors</h2>
    <p><strong>Adapted for NFDI Demonstrator</strong></p>
    <p><strong>Original Workflow:</strong> pyiron_workflow_atomistics team</p>
    <p><strong>Styling:</strong> Based on NFDI Thermal Homogenization Demonstrator</p>
</div>
<div class="funding-info">
    <h2>Funding Acknowledgment</h2>
    <p>TO BE FILLED IN</p>
</div>
"""))

In [ ]:
# --- Grain Boundary Segregation Workflow ---
display(HTML("""
<div class="section">
    <h1>Grain Boundary Segregation Workflow</h1>
    <p>This notebook demonstrates a complete workflow for studying grain boundary (GB) segregation in materials using <code>pyiron_workflow_atomistics</code> and the LAMMPS calculation engine.</p>
    <h2>Overview</h2>
    <ul>
        <li><strong>Lattice Optimization:</strong> Optimize the bulk lattice parameter</li>
        <li><strong>Solution Energy:</strong> Calculate the solution energy of a solute in bulk</li>
        <li><strong>GB Search:</strong> Find suitable grain boundary structures</li>
        <li><strong>Pure GB Study:</strong> Analyze the pure grain boundary properties</li>
        <li><strong>Segregation Study:</strong> Calculate segregation energies at GB sites</li>
    </ul>
</div>
"""))

In [ ]:
import os
from typing import Union, Optional, Tuple

import numpy as np
import pandas as pd

# ASE imports
from ase.build import bulk, stack
from ase.lattice.cubic import BodyCenteredCubic as bcc

# Pyiron workflow imports
import pyiron_workflow as pwf
from pyiron_workflow import Workflow
from pyiron_workflow_atomistics.dataclass_storage import CalcInputMinimize
from pyiron_workflow_atomistics.bulk import optimise_cubic_lattice_parameter
from pyiron_workflow_lammps.engine import LammpsEngine
from pyiron_workflow_atomistics.structure_manipulator.tools import create_supercell, create_supercell_with_min_dimensions
from pyiron_workflow_atomistics.structure_manipulator.tools import substitutional_swap_one_site

from pyiron_workflow_atomistics.gb.gb_code.searcher import get_gb_code_df_with_structures
from pyiron_workflow_atomistics.calculator import calculate_structure_node
# Pymatgen imports
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

%load_ext autoreload
%autoreload 2

In [ ]:
# --- Suppress noisy output ---
import logging
import warnings
import io
import sys

for logger_name in [
    "pyiron_workflow",
    "pyiron_workflow_lammps",
    "pyiron_workflow_atomistics",
    "pyiron_lammps",
    "pyiron_log",       # <-- add this
    "h5io",
]:
    logging.getLogger(logger_name).setLevel(logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)

warnings.filterwarnings("ignore")

class _NullWriter(io.StringIO):
    def write(self, s): pass
    def flush(self): pass

sys.stdout = _NullWriter()

In [ ]:
# --- Verify Installation ---
# display(HTML("""
# <div class="section">
#     <h2>Verify Installation</h2>
#     <p>Let's verify that the <code>pyiron_workflow_atomistics</code> package is correctly installed and accessible.</p>
# </div>
# """))

import pyiron_workflow_atomistics
# print(pyiron_workflow_atomistics.__file__)

In [ ]:
# --- Workflow Initialization ---
# Define the global workflow variable
from pyiron_workflow import Workflow

# Initialize the global workflow
wf = Workflow("gb_segregation", delete_existing_savefiles=True)


In [ ]:
# Define a global Engine for all workflows
from pyiron_workflow_atomistics.dataclass_storage import CalcInputMinimize
from pyiron_workflow_lammps.engine import LammpsEngine

# Initialize the global Engine
inp = CalcInputMinimize()
inp.relax_cell = False  # Don't relax the cell
global_engine = LammpsEngine(EngineInput=inp)
global_engine.working_directory = "gb_segregation"
global_engine.lammps_log_filepath = "minimize.log"
global_engine.command = "lmp -in in.lmp -log minimize.log"
global_engine.input_script_pair_style = "eam/fs"
# potential_path = os.getcwd() + "/Al-Fe.eam.fs"
# potential_path = "/workspace/nfdi-demonstrator/HTSegregation_PyironWorkflow_Demonstrator/Al-Fe.eam.fs"
import pathlib

# Resolves to the directory where this notebook lives, regardless of cwd
_NOTEBOOK_DIR = pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path.cwd()
potential_path = str(_NOTEBOOK_DIR / "Al-Fe.eam.fs")

# Verify it exists and give a clear error early
if not pathlib.Path(potential_path).exists():
    raise FileNotFoundError(
        f"Potential file not found at: {potential_path}\n"
        f"Please place Al-Fe.eam.fs in the same directory as the notebook."
    )
global_engine.path_to_model = potential_path
global_engine.potential_elements = ["Al", "Fe"]


# --- Step 1: Workflow Definition and Lattice Optimization ---
display(HTML("""
<div class="section">
    <h2>Step 1: Workflow Definition and Lattice Optimization</h2>
    <p>In this step, we:</p>
    <ul>
        <li>Initialize the workflow with a name and clean workspace</li>
        <li>Create the initial structure using ASE's bulk builder for Fe (BCC)</li>
        <li>Setup the LAMMPS calculation engine with:
            <ul>
                <li>EAM/FS potential for Al-Fe system</li>
                <li>Minimization settings (no cell relaxation)</li>
                <li>Working directory structure</li>
            </ul>
        </li>
        <li>Optimize the lattice parameter by:
            <ul>
                <li>Applying small strains to the structure</li>
                <li>Calculating energies at each strain</li>
                <li>Fitting an equation of state (Birch-Murnaghan)</li>
                <li>Extracting the equilibrium lattice parameter</li>
            </ul>
        </li>
    </ul>
    <p>This gives us the relaxed bulk structure with the optimal lattice parameter for subsequent calculations.</p>
</div>
"""))



# --- Step 1: Lattice Optimization ---
# Widgets for Step 1
strain_range_widget = widgets.FloatRangeSlider(
    value=(-0.02, 0.02),
    min=-0.1,
    max=0.1,
    step=0.01,
    description='Strain Range:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

num_points_widget = widgets.IntSlider(
    value=10,
    min=5,
    max=50,
    step=1,
    description='Number of Points:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

rattle_widget = widgets.FloatSlider(
    value=0.1,
    min=0.0,
    max=1.0,
    step=0.01,
    description='Rattle:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

# Display widgets for Step 1
display(strain_range_widget, num_points_widget, rattle_widget)

import ipywidgets as widgets
from IPython.display import display

# Create an output widget for Step 1
output_step_1 = widgets.Output()

# def run_step_1(change):
#     from ase.build import bulk
#     from pyiron_workflow_atomistics.bulk import optimise_cubic_lattice_parameter

#     with output_step_1:  # Redirect all output to the output widget
#         output_step_1.clear_output()  # Clear previous output
#         print("Starting Step 1: Lattice Optimization...")
#         structure = bulk("Fe", a=2.85, cubic=True)

#         # Remove the existing node if it already exists
#         if "opt_cubic_cell" in wf.children:
#             print("Removing existing 'opt_cubic_cell' node...")
#             wf.remove_child("opt_cubic_cell")

#         # Optimize the cubic lattice parameter
#         wf.opt_cubic_cell = optimise_cubic_lattice_parameter(
#             structure=structure,
#             name="Fe",
#             crystalstructure="bcc",
#             calculation_engine=global_engine,
#             parent_working_directory="opt_cubic_cell",
#             rattle=rattle_widget.value,
#             strain_range=strain_range_widget.value,
#             num_points=num_points_widget.value,
#             eos_type="birchmurnaghan",
#         )

#         # Run the workflow
#         wf.run()

#         # Debugging: Check and display outputs
#         if hasattr(wf.opt_cubic_cell, "outputs"):
#             print("Step 1 Outputs:")
#             if hasattr(wf.opt_cubic_cell.outputs, "a0"):
#                 print(f"Optimized Lattice Parameter (a0): {wf.opt_cubic_cell.outputs.a0}")
#             if hasattr(wf.opt_cubic_cell.outputs, "equil_volume_per_atom"):
#                 print(f"Equilibrium Volume per Atom: {wf.opt_cubic_cell.outputs.equil_volume_per_atom}")
#         else:
#             print("No outputs available for Step 1.")

#         print("Step 1 completed successfully!")
def run_step_1(change):
    from ase.build import bulk
    from pyiron_workflow_atomistics.bulk import optimise_cubic_lattice_parameter
    import os

    with output_step_1:
        output_step_1.clear_output()

        # Collect parameter info for collapsible
        param_lines = [
            f"Element: Fe (BCC)",
            f"Initial lattice parameter: 2.85 A",
            f"Strain range: {strain_range_widget.value[0]:.3f} to {strain_range_widget.value[1]:.3f}",
            f"Number of strain points: {num_points_widget.value}",
            f"Rattle amplitude: {rattle_widget.value:.3f}",
            f"EOS type: Birch-Murnaghan",
            f"Potential: EAM/FS (Al-Fe)",
            f"Working directory: opt_cubic_cell",
        ]

        display(HTML(f"""
        <details>
        <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px;">
            Analysis Parameters (click to expand)
        </summary>
        <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
            <ul style="margin:0;">
                {"".join(f"<li>{line}</li>" for line in param_lines)}
            </ul>
        </div>
        </details>
        """))

        display(HTML("""
        <div style="background:#e8f4f8; padding:10px; border-radius:5px; margin:10px 0; border-left:4px solid #3498db;">
            Running Step 1: Lattice Optimization. This may take a few moments...
        </div>
        """))

        # # Create and change to calculations directory
        # os.makedirs("calculations", exist_ok=True)
        # os.chdir("calculations")
        
        # Use the notebook's directory as the base
        notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else \
                       os.path.dirname(os.path.abspath(globals().get('__vsc_ipynb_file__', '') or \
                       "/home/gupta/nfdi-demonstrator/HTSegregation_PyironWorkflow_Demonstrator/grain_voila.ipynb"))
        
        calc_dir = os.path.join(notebook_dir, "calculations")
        os.makedirs(calc_dir, exist_ok=True)
        os.chdir(calc_dir)

        # potential_path = "/workspace/nfdi-demonstrator/HTSegregation_PyironWorkflow_Demonstrator/Al-Fe.eam.fs"
        # global_engine.path_to_model = potential_path

        structure = bulk("Fe", a=2.85, cubic=True)

        if "opt_cubic_cell" in wf.children:
            wf.remove_child("opt_cubic_cell")

        wf.opt_cubic_cell = optimise_cubic_lattice_parameter(
            structure=structure,
            name="Fe",
            crystalstructure="bcc",
            calculation_engine=global_engine,
            parent_working_directory="opt_cubic_cell",
            rattle=rattle_widget.value,
            strain_range=strain_range_widget.value,
            num_points=num_points_widget.value,
            eos_type="birchmurnaghan",
        )

        wf.run()

        if hasattr(wf.opt_cubic_cell, "outputs"):
            result_lines = []

            if hasattr(wf.opt_cubic_cell.outputs, "a0"):
                a0 = wf.opt_cubic_cell.outputs.a0.value
                result_lines.append(f"<strong>Optimized Lattice Parameter</strong>")
                result_lines.append(f"a0: {a0:.6f} A")

            if hasattr(wf.opt_cubic_cell.outputs, "equil_volume_per_atom"):
                vol = wf.opt_cubic_cell.outputs.equil_volume_per_atom.value
                result_lines.append(f"Equilibrium volume per atom: {vol:.6f} A^3")

            if hasattr(wf.opt_cubic_cell.outputs, "equil_energy_per_atom"):
                ene = wf.opt_cubic_cell.outputs.equil_energy_per_atom.value
                result_lines.append(f"Equilibrium energy per atom: {ene:.6f} eV")

            if result_lines:
                display(HTML(f"""
                <details open>
                <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                    Step 1 Results (click to expand)
                </summary>
                <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
                    <ul style="margin:0;">
                        {"".join(f"<li>{line}</li>" for line in result_lines)}
                    </ul>
                </div>
                </details>
                """))
            else:
                display(HTML("""
                <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                    <strong>Warning:</strong> No output values available from lattice optimization.
                </div>
                """))
        else:
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Warning:</strong> No outputs available for Step 1.
            </div>
            """))

        display(HTML("""
        <div style="background:#e6f4ea; padding:10px; border-radius:5px; margin-top:10px; border-left:4px solid #2e7d32;">
            Step 1 completed successfully. Optimized structure is ready for subsequent steps.
        </div>
        """))

# Add a button to trigger Step 1
run_button_step_1 = widgets.Button(description="Run Step 1")
run_button_step_1.on_click(run_step_1)

# Display the button and the output area
display(run_button_step_1, output_step_1)

In [ ]:
# --- Step 2: Solution Energy Calculation ---
display(HTML("""
<div class="section">
    <h2>Step 2: Solution Energy Calculation</h2>
    <p>The solution energy tells us how favorable it is to dissolve a solute atom (Al) into the bulk host material (Fe).</p>
    <p>In this step, we:</p>
    <ul>
        <li>Create a large supercell (12x12x12 Å minimum dimensions) to avoid spurious interactions between periodic images of the solute</li>
        <li>Calculate the energy of pure bulk supercell</li>
        <li>Create a supercell with one Al solute by substituting one Fe atom at position 0</li>
        <li>Calculate the energy with the solute present</li>
        <li>Compute the solution energy as: <br>
            <span style="font-family:monospace;">E_solution = E_with_solute - E_pure_bulk</span>
        </li>
    </ul>
    <p>A positive solution energy indicates that dissolving Al in Fe is energetically unfavorable in the bulk.</p>
    <p>This value will be used later to calculate segregation energies at grain boundaries.</p>
</div>
"""))

from pyiron_workflow_atomistics.utils import duplicate_engine


# --- Step 2: Solution Energy Calculation ---
# Widgets for Step 2
min_dimensions_widget = widgets.FloatSlider(
    value=12.0,
    min=10.0,
    max=20.0,
    step=0.5,
    description='Min Dimensions:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

defect_site_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=100,
    step=1,
    description='Defect Site:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

# Display widgets for Step 2
display(min_dimensions_widget, defect_site_widget)

output_step_2 = widgets.Output()

def run_step_2(change):
    from pyiron_workflow_atomistics.structure_manipulator.tools import (
        create_supercell_with_min_dimensions,
        substitutional_swap_one_site,
    )
    from pyiron_workflow_atomistics.calculator import calculate_structure_node
    from pyiron_workflow_atomistics.dataclass_storage import CalcInputMinimize
    from pyiron_workflow_lammps.engine import LammpsEngine

    with output_step_2:
        output_step_2.clear_output()

        # Check if Step 1 outputs exist
        if not hasattr(wf.opt_cubic_cell.outputs, "equil_struct"):
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Error:</strong> Step 1 outputs are missing. Please run Step 1 first.
            </div>
            """))
            return

        # Collect parameter info for collapsible
        param_lines = [
            f"Host element: Fe (BCC)",
            f"Solute element: Al",
            f"Supercell min dimensions: {min_dimensions_widget.value:.1f} x {min_dimensions_widget.value:.1f} x {min_dimensions_widget.value:.1f} A",
            f"Defect substitution site index: {defect_site_widget.value}",
            f"Potential: EAM/FS (Al-Fe)",
            f"Working directory (bulk): solution_energy_bulk",
            f"Working directory (solute): solution_energy_solute",
        ]

        display(HTML(f"""
        <details>
        <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px;">
            Analysis Parameters (click to expand)
        </summary>
        <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
            <ul style="margin:0;">
                {"".join(f"<li>{line}</li>" for line in param_lines)}
            </ul>
        </div>
        </details>
        """))

        display(HTML("""
        <div style="background:#e8f4f8; padding:10px; border-radius:5px; margin:10px 0; border-left:4px solid #3498db;">
            Running Step 2: Solution Energy Calculation. This may take a few moments...
        </div>
        """))

        # Remove existing nodes if they already exist
        for node in ["supercell", "supercell_calc", "supercell_with_1sol", "supercell_with_1sol_calc"]:
            if node in wf.children:
                wf.remove_child(node)

        # Create FRESH engine for bulk calculations
        inp_bulk = CalcInputMinimize()
        inp_bulk.relax_cell = False
        Engine_bulk = LammpsEngine(EngineInput=inp_bulk)
        Engine_bulk.working_directory = "solution_energy_bulk"
        Engine_bulk.lammps_log_filepath = "minimize.log"
        Engine_bulk.command = "lmp -in in.lmp -log minimize.log"
        Engine_bulk.input_script_pair_style = "eam/fs"
        Engine_bulk.path_to_model = potential_path
        Engine_bulk.potential_elements = ["Al", "Fe"]

        # Create supercell using Step 1 output structure
        wf.supercell = create_supercell_with_min_dimensions(
            base_structure=wf.opt_cubic_cell.outputs.equil_struct,
            min_dimensions=[min_dimensions_widget.value] * 3,
        )
        wf.supercell_calc = calculate_structure_node(wf.supercell, calculation_engine=Engine_bulk)

        # Create FRESH engine for solute calculations
        inp_bulk_solute = CalcInputMinimize()
        inp_bulk_solute.relax_cell = False
        Engine_bulk_solute = LammpsEngine(EngineInput=inp_bulk_solute)
        Engine_bulk_solute.working_directory = "solution_energy_solute"
        Engine_bulk_solute.lammps_log_filepath = "minimize.log"
        Engine_bulk_solute.command = "lmp -in in.lmp -log minimize.log"
        Engine_bulk_solute.input_script_pair_style = "eam/fs"
        Engine_bulk_solute.path_to_model = potential_path
        Engine_bulk_solute.potential_elements = ["Al", "Fe"]

        wf.supercell_with_1sol = substitutional_swap_one_site(
            base_structure=wf.supercell,
            defect_site=defect_site_widget.value,
            new_symbol="Al",
        )
        wf.supercell_with_1sol_calc = calculate_structure_node(
            structure=wf.supercell_with_1sol,
            calculation_engine=Engine_bulk_solute,
        )

        wf.run()

        result_lines = []

        if hasattr(wf.supercell_calc.outputs, "calc_output"):
            supercell_struct = wf.supercell.value
            calc_output = wf.supercell_calc.outputs.calc_output.value
            result_lines.append(f"<strong>Pure Bulk Supercell</strong>")
            result_lines.append(f"Number of atoms: {len(supercell_struct)}")
            result_lines.append(f"Composition: {supercell_struct.get_chemical_formula()}")
            result_lines.append(f"Total energy: {calc_output.final_energy:.6f} eV")

        if hasattr(wf.supercell_with_1sol_calc.outputs, "calc_output"):
            supercell_sol_struct = wf.supercell_with_1sol.value
            calc_output_sol = wf.supercell_with_1sol_calc.outputs.calc_output.value
            result_lines.append(f"<strong>Supercell with Al Substitution</strong>")
            result_lines.append(f"Number of atoms: {len(supercell_sol_struct)}")
            result_lines.append(f"Composition: {supercell_sol_struct.get_chemical_formula()}")
            result_lines.append(f"Total energy: {calc_output_sol.final_energy:.6f} eV")

            if hasattr(wf.supercell_calc.outputs, "calc_output"):
                pure_energy = wf.supercell_calc.outputs.calc_output.value.final_energy
                solute_energy = calc_output_sol.final_energy
                solution_energy = solute_energy - pure_energy

                result_lines.append(f"<strong>Solution Energy</strong>")
                result_lines.append(f"E(pure bulk): {pure_energy:.6f} eV")
                result_lines.append(f"E(with Al): {solute_energy:.6f} eV")
                result_lines.append(f"dE = E(with Al) - E(pure): {solution_energy:.6f} eV")

                # Create workflow node for solution energy (needed for Step 5)
                if "soln_energy" in wf.children:
                    wf.remove_child("soln_energy")

                @pwf.as_function_node("solution_energy")
                def calculate_soln_energy(bulk_structure_energy, soln_structure_energy):
                    return soln_structure_energy - bulk_structure_energy

                wf.soln_energy = calculate_soln_energy(
                    bulk_structure_energy=wf.supercell_calc.outputs.calc_output.final_energy,
                    soln_structure_energy=wf.supercell_with_1sol_calc.outputs.calc_output.final_energy,
                )

        if result_lines:
            display(HTML(f"""
            <details open>
            <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                Step 2 Results (click to expand)
            </summary>
            <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
                <ul style="margin:0;">
                    {"".join(f"<li>{line}</li>" for line in result_lines)}
                </ul>
            </div>
            </details>
            """))
        else:
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Warning:</strong> No output values available from solution energy calculation.
            </div>
            """))

        display(HTML("""
        <div style="background:#e6f4ea; padding:10px; border-radius:5px; margin-top:10px; border-left:4px solid #2e7d32;">
            Step 2 completed successfully. Solution energy is stored in wf.soln_energy for Step 5.
        </div>
        """))
        
# Add a button to trigger Step 2
run_button_step_2 = widgets.Button(description="Run Step 2")
run_button_step_2.on_click(run_step_2)

display(run_button_step_2, output_step_2)

In [ ]:
# # --- Step 3: Grain Boundary Search ---
# display(HTML("""
# <div class="section">
#     <h2>Step 3: Grain Boundary Search</h2>
#     <p>Now we search for suitable grain boundary structures using the GBCode algorithm.</p>
#     <p>In this step, we:</p>
#     <ul>
#         <li>Search for CSL (Coincidence Site Lattice) grain boundaries along three different crystallographic axes: [111], [110], and [100]</li>
#         <li>Filter structures based on:
#             <ul>
#                 <li>Sigma values (up to Σ10)</li>
#                 <li>Maximum number of atoms (≤100)</li>
#                 <li>Minimum grain dimensions (10 Å in-plane, 15 Å along grain)</li>
#             </ul>
#         </li>
#         <li>Generate atomistic structures for each valid GB configuration</li>
#         <li>Remove duplicates to get a unique set of GB structures</li>
#     </ul>
#     <p>The output is a DataFrame containing various GB structures with different misorientations, planes, and atomic arrangements.</p>
#     <p>For this workflow, we'll select the first GB structure from the results for detailed analysis.</p>
# </div>
# """))

# # Search GBCode for valid CSL GB structures
# wf.gb_code_df = get_gb_code_df_with_structures(
#     axes_list=[np.array([1, 1, 1]), np.array([1, 1, 0]), np.array([1, 0, 0])],
#     sigma_limit=10,
#     lim_plane_index=3,
#     max_atoms=100,
#     max_workers=None,
#     deduplicate=True,
#     element="Fe",
#     basis="bcc",
#     lattice_param=wf.opt_cubic_cell.outputs.a0,
#     equil_volume_per_atom=wf.opt_cubic_cell.outputs.equil_volume_per_atom,
#     min_inplane_gb_length=10,
#     req_length_grain=15,
#     grain_length_axis=0,
# )

# --- Step 3: Grain Boundary Search ---
display(HTML("""
<div class="section">
    <h2>Step 3: Grain Boundary Search</h2>
    <p>Now we search for suitable grain boundary structures using the GBCode algorithm.</p>
    <p>In this step, we:</p>
    <ul>
        <li>Search for CSL (Coincidence Site Lattice) grain boundaries along three different crystallographic axes: [111], [110], and [100]</li>
        <li>Filter structures based on:
            <ul>
                <li>Sigma values (up to Σ10)</li>
                <li>Maximum number of atoms (≤100)</li>
                <li>Minimum grain dimensions (10 Å in-plane, 15 Å along grain)</li>
            </ul>
        </li>
        <li>Generate atomistic structures for each valid GB configuration</li>
        <li>Remove duplicates to get a unique set of GB structures</li>
    </ul>
    <p>The output is a DataFrame containing various GB structures with different misorientations, planes, and atomic arrangements.</p>
    <p>For this workflow, we'll select the first GB structure from the results for detailed analysis.</p>
</div>
"""))

# # Widgets for Step 3
# sigma_limit_widget = widgets.IntSlider(
#     value=10,
#     min=1,
#     max=20,
#     step=1,
#     description='Sigma Limit:',
#     continuous_update=False,
#     style={'description_width': 'initial'}
# )

# max_atoms_widget = widgets.IntSlider(
#     value=100,
#     min=10,
#     max=500,
#     step=10,
#     description='Max Atoms:',
#     continuous_update=False,
#     style={'description_width': 'initial'}
# )

# min_inplane_gb_length_widget = widgets.FloatSlider(
#     value=10.0,
#     min=5.0,
#     max=20.0,
#     step=0.5,
#     description='Min In-Plane Length:',
#     continuous_update=False,
#     style={'description_width': 'initial'}
# )

# req_length_grain_widget = widgets.FloatSlider(
#     value=15.0,
#     min=10.0,
#     max=30.0,
#     step=0.5,
#     description='Required Grain Length:',
#     continuous_update=False,
#     style={'description_width': 'initial'}
# )

# # Display widgets
# display(sigma_limit_widget, max_atoms_widget, min_inplane_gb_length_widget, req_length_grain_widget)

# # Function to run Step 3 workflow
# def run_step_3(change):
#     # Search GBCode for valid CSL GB structures
#     wf.gb_code_df = get_gb_code_df_with_structures(
#         axes_list=[np.array([1, 1, 1]), np.array([1, 1, 0]), np.array([1, 0, 0])],
#         sigma_limit=sigma_limit_widget.value,
#         lim_plane_index=3,
#         max_atoms=max_atoms_widget.value,
#         max_workers=None,
#         deduplicate=True,
#         element="Fe",
#         basis="bcc",
#         lattice_param=wf.opt_cubic_cell.outputs.a0,
#         equil_volume_per_atom=wf.opt_cubic_cell.outputs.equil_volume_per_atom,
#         min_inplane_gb_length=min_inplane_gb_length_widget.value,
#         req_length_grain=req_length_grain_widget.value,
#         grain_length_axis=0,
#     )

#     # Display the DataFrame and plots
#     if hasattr(wf.gb_code_df.outputs, "gb_code_df_with_structures"):
#         gb_df = wf.gb_code_df.outputs.gb_code_df_with_structures
#         display(HTML("<h3>Grain Boundary Structures DataFrame:</h3>"))
#         display(gb_df)

#         # Plot the first GB structure if available
#         if not gb_df.empty:
#             first_gb_structure = gb_df.iloc[0].structure
#             if hasattr(first_gb_structure, "plot"):
#                 print("Plotting the first GB structure:")
#                 display(first_gb_structure.plot())
#             else:
#                 print("No plot available for the first GB structure.")
#         else:
#             print("No grain boundary structures found.")
#     else:
#         print("No output DataFrame available.")

# # Add a button to trigger Step 3
# run_button_step_3 = widgets.Button(description="Run Step 3: Grain Boundary Search")
# run_button_step_3.on_click(run_step_3)
# display(run_button_step_3)

# --- Step 3: Grain Boundary Search ---
# Widgets for Step 3
sigma_limit_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=20,
    step=1,
    description='Sigma Limit:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

max_atoms_widget = widgets.IntSlider(
    value=100,
    min=10,
    max=500,
    step=10,
    description='Max Atoms:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

min_inplane_gb_length_widget = widgets.FloatSlider(
    value=10.0,
    min=5.0,
    max=20.0,
    step=0.5,
    description='Min In-Plane Length:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

req_length_grain_widget = widgets.FloatSlider(
    value=15.0,
    min=10.0,
    max=30.0,
    step=0.5,
    description='Required Grain Length:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

# Display widgets for Step 3
display(sigma_limit_widget, max_atoms_widget, min_inplane_gb_length_widget, req_length_grain_widget)

output_step_3 = widgets.Output()

def run_step_3(change):
    from pyiron_workflow_atomistics.gb.gb_code.searcher import get_gb_code_df_with_structures
    import pyiron_workflow_atomistics.gb.gb_code.searcher as _searcher_mod
    import tqdm
    import tqdm.auto
    import tqdm.notebook

    with output_step_3:
        output_step_3.clear_output()

        # Check if Step 1 outputs exist
        if not hasattr(wf.opt_cubic_cell.outputs, "a0"):
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Error:</strong> Step 1 outputs are missing. Please run Step 1 first.
            </div>
            """))
            return

        # Collect parameter info for collapsible
        param_lines = [
            f"Element: Fe (BCC)",
            f"Lattice parameter from Step 1: {wf.opt_cubic_cell.outputs.a0.value:.4f} Å",
            f"Search axes: [111], [110], [100]",
            f"Sigma limit: \u03a3{sigma_limit_widget.value}",
            f"Max atoms per structure: {max_atoms_widget.value}",
            f"Min in-plane GB length: {min_inplane_gb_length_widget.value} Å",
            f"Required grain length: {req_length_grain_widget.value} Å",
            f"Plane index limit: 3",
            f"Deduplicate structures: True",
        ]

        display(HTML(f"""
        <details>
        <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px;">
            Analysis Parameters (click to expand)
        </summary>
        <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
            <ul style="margin:0;">
                {"".join(f"<li>{line}</li>" for line in param_lines)}
            </ul>
        </div>
        </details>
        """))

        display(HTML("""
        <div style="background:#e8f4f8; padding:10px; border-radius:5px; margin:10px 0; border-left:4px solid #3498db;">
            Running Step 3: Grain Boundary Search. Progress bars will appear below...
        </div>
        <div style="font-weight:bold; padding:6px 0 4px 0; color:#2c3e50;">Search Progress:</div>
        """))

        # Save originals
        original_tqdm = tqdm.tqdm
        original_tqdm_auto = tqdm.auto.tqdm
        original_searcher_tqdm = getattr(_searcher_mod, 'tqdm', None)

        # Patch tqdm everywhere: top-level, tqdm.auto, and directly inside the searcher module
        tqdm.tqdm = tqdm.notebook.tqdm
        tqdm.auto.tqdm = tqdm.notebook.tqdm
        if original_searcher_tqdm is not None:
            _searcher_mod.tqdm = tqdm.notebook.tqdm

        try:
            # Remove existing node if it exists
            if "gb_code_df" in wf.children:
                wf.remove_child("gb_code_df")

            wf.gb_code_df = get_gb_code_df_with_structures(
                axes_list=[np.array([1, 1, 1]), np.array([1, 1, 0]), np.array([1, 0, 0])],
                sigma_limit=sigma_limit_widget.value,
                lim_plane_index=3,
                max_atoms=max_atoms_widget.value,
                max_workers=None,
                deduplicate=True,
                element="Fe",
                basis="bcc",
                lattice_param=wf.opt_cubic_cell.outputs.a0,
                equil_volume_per_atom=wf.opt_cubic_cell.outputs.equil_volume_per_atom,
                min_inplane_gb_length=min_inplane_gb_length_widget.value,
                req_length_grain=req_length_grain_widget.value,
                grain_length_axis=0,
            )
            wf.run()
        finally:
            # Always restore originals
            tqdm.tqdm = original_tqdm
            tqdm.auto.tqdm = original_tqdm_auto
            if original_searcher_tqdm is not None:
                _searcher_mod.tqdm = original_searcher_tqdm

        if hasattr(wf.gb_code_df.outputs, "gb_code_df_with_structures"):
            gb_df = wf.gb_code_df.outputs.gb_code_df_with_structures.value
            result_lines = []

            result_lines.append(f"<strong>Search Summary</strong>")
            result_lines.append(f"Total GB structures found: {len(gb_df)}")

            if len(gb_df) > 0:
                if 'Sigma' in gb_df.columns:
                    result_lines.append(f"Sigma values found: {sorted(gb_df['Sigma'].unique())}")
                elif 'sigma' in gb_df.columns:
                    result_lines.append(f"Sigma values found: {sorted(gb_df['sigma'].unique())}")

                if 'n_atoms' in gb_df.columns:
                    result_lines.append(f"Atom count range: {gb_df['n_atoms'].min()} – {gb_df['n_atoms'].max()}")
                elif 'natoms' in gb_df.columns:
                    result_lines.append(f"Atom count range: {gb_df['natoms'].min()} – {gb_df['natoms'].max()}")

                first_gb = gb_df.iloc[0]
                result_lines.append(f"<strong>First GB Structure (selected for Step 4)</strong>")
                for col in gb_df.columns[:10]:
                    if col != 'structure':
                        result_lines.append(f"{col}: {first_gb[col]}")

                display(HTML(f"""
                <details open>
                <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                    Step 3 Results (click to expand)
                </summary>
                <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
                    <ul style="margin:0;">
                        {"".join(f"<li>{line}</li>" for line in result_lines)}
                    </ul>
                </div>
                </details>
                """))

                priority_cols = ['Sigma', 'Σ', 'sigma', 'Type', 'Axis', 'axis', 'plane', 'n_atoms', 'natoms', 'rotation_angle']
                display_cols = [col for col in priority_cols if col in gb_df.columns]
                if not display_cols:
                    display_cols = [col for col in gb_df.columns if col != 'structure'][:6]

                display(HTML(f"""
                <details>
                <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                    GB Structures Table — first 10 rows (click to expand)
                </summary>
                <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db; overflow-x:auto;">
                    {gb_df[display_cols].head(10).to_html(index=False, border=0, classes='table')}
                </div>
                </details>
                """))

            else:
                display(HTML("""
                <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                    <strong>Warning:</strong> No grain boundary structures found. Try increasing max_atoms or sigma_limit.
                </div>
                """))
                return

        else:
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Warning:</strong> No output DataFrame available from GB search.
            </div>
            """))
            return

        display(HTML("""
        <div style="background:#e6f4ea; padding:10px; border-radius:5px; margin-top:10px; border-left:4px solid #2e7d32;">
            Step 3 completed successfully. GB structures are stored in wf.gb_code_df for Step 4.
        </div>
        """))
# Add a button to trigger Step 3
run_button_step_3 = widgets.Button(description="Run Step 3")
run_button_step_3.on_click(run_step_3)

display(run_button_step_3, output_step_3)

In [ ]:
# # --- Step 4: Pure Grain Boundary Study ---
# display(HTML("""
# <div class="section">
#     <h2>Step 4: Pure Grain Boundary Study</h2>
#     <p>This comprehensive step analyzes the properties of the pure (undecorated) grain boundary.</p>
#     <ul>
#         <li><strong>GB Plane Identification:</strong> Uses Voronoi site featurization to identify atomic environments, compares GB region atoms to bulk templates, locates the GB plane position automatically.</li>
#         <li><strong>Grain Length Optimization:</strong> Coarse and fine scan to find the optimal grain extension that minimizes GB energy.</li>
#         <li><strong>Cleavage Energy Calculation:</strong> Identifies viable cleavage planes near the GB, calculates rigid and relaxed cleavage energy.</li>
#         <li><strong>Structure Preparation:</strong> Adds vacuum layer (20 Å) for surface calculations, expands cell to minimum dimensions (6×6 Å in-plane), prepares the final GB structure for segregation studies.</li>
#     </ul>
#     <p><strong>Output:</strong> Optimized pure GB structure with known GB plane location, GB energy, and cleavage properties.</p>
# </div>
# """))

# from pyiron_workflow_atomistics.dataclass_storage import CalcInputStatic
# from pyiron_workflow_atomistics.gb.gb_study import pure_gb_study
# from pyiron_workflow_atomistics.gb.dataclass_storage import CleaveGBStructureInput, PlotCleaveInput
# from pyiron_workflow_atomistics.featurisers import voronoiSiteFeaturiser

# # Create engines for GB study
# Engine_gb = duplicate_engine.node_function(Engine, "gb_study")
# Engine_gb.working_directory = "gb_study"

# # Static engine for cleavage energy calculations
# inp_static = CalcInputStatic()
# Engine_static = LammpsEngine(EngineInput=inp_static)
# Engine_static.working_directory = "pure_grain_boundary_study"
# Engine_static.lammps_log_filepath = "static.log"
# Engine_static.command = "lmp -in in.lmp -log static.log"
# Engine_static.input_script_pair_style = "eam/fs"
# Engine_static.path_to_model = potential_path

# # Run pure GB study
# wf.pure_gb_study = pure_gb_study(
#     gb_structure=wf.gb_code_df.outputs.gb_code_df_with_structures.iloc[0].structure,
#     equil_bulk_volume=wf.opt_cubic_cell.outputs.equil_volume_per_atom,
#     equil_bulk_energy=wf.opt_cubic_cell.outputs.equil_energy_per_atom,
#     extensions_stage1=np.linspace(-0.2, 0.8, 3),
#     extensions_stage2=np.linspace(-0.05, 0.05, 5),
#     calculation_engine=Engine_gb,
#     static_calculation_engine=Engine_static,
#     length_interpolate_min_n_points=5,
#     gb_normal_axis="c",
#     vacuum_length=20,
#     min_inplane_cell_lengths=[6, 6, None],
#     featuriser=voronoiSiteFeaturiser,
#     approx_frac=0.5,
#     tolerance=5.0,
#     bulk_offset=10.0,
#     slab_thickness=2.0,
#     featuriser_kwargs=None,
#     n_bulk=10,
#     threshold_frac=0.3,
#     CleaveGBStructure_Input=CleaveGBStructureInput(tol=0.3),
#     PlotCleave_Input=PlotCleaveInput()
# )

# --- Step 4: Pure Grain Boundary Study ---
display(HTML("""
<div class="section">
    <h2>Step 4: Pure Grain Boundary Study</h2>
    <p>This comprehensive step analyzes the properties of the pure (undecorated) grain boundary.</p>
    <ul>
        <li><strong>GB Plane Identification:</strong> Uses Voronoi site featurization to identify atomic environments, compares GB region atoms to bulk templates, locates the GB plane position automatically.</li>
        <li><strong>Grain Length Optimization:</strong> Coarse and fine scan to find the optimal grain extension that minimizes GB energy.</li>
        <li><strong>Cleavage Energy Calculation:</strong> Identifies viable cleavage planes near the GB, calculates rigid and relaxed cleavage energy.</li>
        <li><strong>Structure Preparation:</strong> Adds vacuum layer (20 Å) for surface calculations, expands cell to minimum dimensions (6×6 Å in-plane), prepares the final GB structure for segregation studies.</li>
    </ul>
    <p><strong>Output:</strong> Optimized pure GB structure with known GB plane location, GB energy, and cleavage properties.</p>
</div>
"""))

# Widgets for Step 4
extensions_stage1_widget = widgets.FloatRangeSlider(
    value=(-0.2, 0.8),
    min=-1.0,
    max=1.0,
    step=0.1,
    description='Extensions Stage 1:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

extensions_stage2_widget = widgets.FloatRangeSlider(
    value=(-0.05, 0.05),
    min=-0.1,
    max=0.1,
    step=0.01,
    description='Extensions Stage 2:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

vacuum_length_widget = widgets.FloatSlider(
    value=20.0,
    min=10.0,
    max=50.0,
    step=1.0,
    description='Vacuum Length:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

min_inplane_cell_lengths_widget = widgets.FloatSlider(
    value=6.0,
    min=3.0,
    max=10.0,
    step=0.5,
    description='Min In-Plane Cell Length:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

# Display widgets
display(extensions_stage1_widget, extensions_stage2_widget, vacuum_length_widget, min_inplane_cell_lengths_widget)

output_step_4 = widgets.Output()

def run_step_4(change):
    from pyiron_workflow_atomistics.dataclass_storage import CalcInputStatic
    from pyiron_workflow_atomistics.gb.gb_study import pure_gb_study
    from pyiron_workflow_atomistics.gb.dataclass_storage import CleaveGBStructureInput, PlotCleaveInput
    from pyiron_workflow_atomistics.featurisers import voronoiSiteFeaturiser
    from pyiron_workflow_lammps.engine import LammpsEngine
    import io

    with output_step_4:
        output_step_4.clear_output()

        # Check if Step 3 outputs exist
        if not hasattr(wf.gb_code_df.outputs, "gb_code_df_with_structures"):
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Error:</strong> Step 3 outputs are missing. Please run Step 3 first.
            </div>
            """))
            return

        gb_df = wf.gb_code_df.outputs.gb_code_df_with_structures.value
        first_gb = gb_df.iloc[0]

        # Collect parameter info for collapsible
        param_lines = []
        param_lines.append(f"Selected GB structure: First structure from Step 3")
        param_lines.append(f"GB type: Sigma{first_gb['Sigma']} {first_gb['Type']}")
        param_lines.append(f"Axis: {first_gb['Axis']}")
        param_lines.append(f"Initial atoms: {first_gb['n_atoms']}")
        param_lines.append(f"Grain length extensions (stage 1): {extensions_stage1_widget.value[0]:.2f} to {extensions_stage1_widget.value[1]:.2f}")
        param_lines.append(f"Grain length extensions (stage 2): {extensions_stage2_widget.value[0]:.3f} to {extensions_stage2_widget.value[1]:.3f}")
        param_lines.append(f"Vacuum length: {vacuum_length_widget.value} A")
        param_lines.append(f"Min in-plane cell length: {min_inplane_cell_lengths_widget.value} A")

        display(HTML(f"""
        <details>
        <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px;">
            Analysis Parameters (click to expand)
        </summary>
        <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
            <ul style="margin:0;">
                {"".join(f"<li>{line}</li>" for line in param_lines)}
            </ul>
        </div>
        </details>
        """))

        display(HTML("""
        <div style="background:#e8f4f8; padding:10px; border-radius:5px; margin:10px 0; border-left:4px solid #3498db;">
            Running Step 4: Pure Grain Boundary Study. This may take several minutes...
        </div>
        """))

        # Remove existing node if it exists
        if "pure_gb_study" in wf.children:
            wf.remove_child("pure_gb_study")

        # Create engines
        inp_gb = CalcInputMinimize()
        inp_gb.relax_cell = False
        Engine_gb = LammpsEngine(EngineInput=inp_gb)
        Engine_gb.working_directory = "gb_study"
        Engine_gb.lammps_log_filepath = "minimize.log"
        Engine_gb.command = "lmp -in in.lmp -log minimize.log"
        Engine_gb.input_script_pair_style = "eam/fs"
        Engine_gb.path_to_model = potential_path
        Engine_gb.potential_elements = ["Al", "Fe"]

        inp_static = CalcInputStatic()
        Engine_static = LammpsEngine(EngineInput=inp_static)
        Engine_static.working_directory = "pure_grain_boundary_study"
        Engine_static.lammps_log_filepath = "static.log"
        Engine_static.command = "lmp -in in.lmp -log static.log"
        Engine_static.input_script_pair_style = "eam/fs"
        Engine_static.path_to_model = potential_path
        Engine_static.potential_elements = ["Al", "Fe"]

        wf.pure_gb_study = pure_gb_study(
            gb_structure=wf.gb_code_df.outputs.gb_code_df_with_structures.value.iloc[0].structure,
            equil_bulk_volume=wf.opt_cubic_cell.outputs.equil_volume_per_atom,
            equil_bulk_energy=wf.opt_cubic_cell.outputs.equil_energy_per_atom,
            extensions_stage1=np.linspace(*extensions_stage1_widget.value, 3),
            extensions_stage2=np.linspace(*extensions_stage2_widget.value, 5),
            calculation_engine=Engine_gb,
            static_calculation_engine=Engine_static,
            length_interpolate_min_n_points=5,
            gb_normal_axis="c",
            vacuum_length=vacuum_length_widget.value,
            min_inplane_cell_lengths=[min_inplane_cell_lengths_widget.value] * 2 + [None],
            featuriser=voronoiSiteFeaturiser,
            approx_frac=0.5,
            tolerance=5.0,
            bulk_offset=10.0,
            slab_thickness=2.0,
            featuriser_kwargs=None,
            n_bulk=10,
            threshold_frac=0.3,
            CleaveGBStructure_Input=CleaveGBStructureInput(tol=0.3),
            PlotCleave_Input=PlotCleaveInput()
        )

        wf.run()

        # Display cleavage plots
        import matplotlib.pyplot as plt
        fignums = plt.get_fignums()
        if fignums:
            display(HTML("""
            <div style="font-weight:bold; margin:10px 0 5px 0; color:#2c3e50;">
                Cleavage Plane Visualizations:
            </div>
            """))
            for num in fignums:
                fig = plt.figure(num)
                display(fig)

        # Results summary
        if hasattr(wf.pure_gb_study, "outputs"):
            result_lines = []

            if hasattr(wf.pure_gb_study.outputs, "gb_plane_analysis_dict"):
                gb_analysis = wf.pure_gb_study.outputs.gb_plane_analysis_dict.value
                result_lines.append(f"<strong>GB Plane Identification</strong>")
                result_lines.append(f"GB plane position (Cartesian): {gb_analysis['gb_cart']:.4f} A")
                result_lines.append(f"GB plane position (Fractional): {gb_analysis['gb_frac']:.4f}")
                result_lines.append(f"GB region atoms identified: {len(gb_analysis['extended_sel_indices'])}")

            if hasattr(wf.pure_gb_study.outputs, "pure_grain_boundary_structure_vacuum"):
                gb_struct = wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum.value
                result_lines.append(f"<strong>Optimized GB Structure</strong>")
                result_lines.append(f"Total atoms: {len(gb_struct)}")
                result_lines.append(f"Cell dimensions: [{gb_struct.cell[0][0]:.2f}, {gb_struct.cell[1][1]:.2f}, {gb_struct.cell[2][2]:.2f}] A")
                result_lines.append(f"Composition: {gb_struct.get_chemical_formula()}")

            if hasattr(wf.pure_gb_study.outputs, "grain_boundary_energy"):
                gb_energy = wf.pure_gb_study.outputs.grain_boundary_energy.value
                result_lines.append(f"<strong>GB Energetics</strong>")
                result_lines.append(f"GB energy: {gb_energy:.4f} J/m2")

                if hasattr(wf.pure_gb_study.outputs, "pure_grain_boundary_structure_vacuum_energy"):
                    gb_total_energy = wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum_energy.value
                    result_lines.append(f"Total energy (with vacuum): {gb_total_energy:.4f} eV")

                if hasattr(wf.pure_gb_study.outputs, "grain_boundary_excess_volume"):
                    excess_vol = wf.pure_gb_study.outputs.grain_boundary_excess_volume.value
                    result_lines.append(f"GB excess volume: {excess_vol:.4f} A3/A2")

            if hasattr(wf.pure_gb_study.outputs, "work_of_separation_rigid"):
                w_sep_rigid = wf.pure_gb_study.outputs.work_of_separation_rigid.value
                result_lines.append(f"<strong>Cleavage Properties</strong>")
                result_lines.append(f"Work of separation (rigid): {w_sep_rigid:.4f} J/m2")

                if hasattr(wf.pure_gb_study.outputs, "work_of_separation_relaxed"):
                    w_sep_relaxed = wf.pure_gb_study.outputs.work_of_separation_relaxed.value
                    result_lines.append(f"Work of separation (relaxed): {w_sep_relaxed:.4f} J/m2")
                    result_lines.append(f"Relaxation energy gain: {w_sep_rigid - w_sep_relaxed:.4f} J/m2")

                if hasattr(wf.pure_gb_study.outputs, "surface_energy"):
                    surf_energy = wf.pure_gb_study.outputs.surface_energy.value
                    result_lines.append(f"Surface energy: {surf_energy:.4f} J/m2")

            if hasattr(wf.pure_gb_study.outputs, "grain_boundary_length_optimisation_df"):
                opt_df = wf.pure_gb_study.outputs.grain_boundary_length_optimisation_df.value
                result_lines.append(f"<strong>Grain Length Optimization</strong>")
                result_lines.append(f"Structures tested: {len(opt_df)}")
                if 'gb_energy' in opt_df.columns:
                    result_lines.append(f"GB energy range: {opt_df['gb_energy'].min():.4f} to {opt_df['gb_energy'].max():.4f} J/m2")

            display(HTML(f"""
            <details>
            <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                Step 4 Results (click to expand)
            </summary>
            <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
                <ul style="margin:0;">
                    {"".join(f"<li>{line}</li>" for line in result_lines)}
                </ul>
            </div>
            </details>
            """))

            display(HTML("""
            <div style="background:#e6f4ea; padding:10px; border-radius:5px; margin-top:10px; border-left:4px solid #2e7d32;">
                Step 4 completed successfully. Pure GB structure is ready for Step 5.
            </div>
            """))
        else:
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Warning:</strong> No outputs available from pure_gb_study.
            </div>
            """))

# Add a button to trigger Step 4
run_button_step_4 = widgets.Button(description="Run Step 4")
run_button_step_4.on_click(run_step_4)

display(run_button_step_4, output_step_4)

In [ ]:
# --- Step 5: Segregation Study ---
display(HTML("""
<div class="section">
    <h2>Step 5: Segregation Study</h2>
    <p>This is the main segregation calculation where we determine which GB sites are energetically favorable for Al segregation.</p>
    <ul>
        <li><strong>Site Identification and Deduplication:</strong> Use SOAP descriptors and PCA to group sites with similar environments and select representatives.</li>
        <li><strong>Segregation Energy Calculations:</strong>
            <ul>
                <li>Create a structure with Al substituted at each unique site</li>
                <li>Calculate the total energy using LAMMPS</li>
                <li>Compute segregation energy as:<br>
                    <span style="font-family:monospace;">E_seg = (E_GB+Al - E_GB_pure) - E_solution</span>
                </li>
            </ul>
        </li>
        <li><strong>Interpretation:</strong>
            <ul>
                <li>Negative E_seg: Segregation is favorable (Al prefers GB over bulk)</li>
                <li>Positive E_seg: Segregation is unfavorable (Al prefers bulk)</li>
            </ul>
        </li>
    </ul>
    <p>The magnitude indicates the strength of the segregation tendency.</p>
</div>
"""))

# Widgets for Step 5
r_cut_widget = widgets.FloatSlider(
    value=6.0,
    min=3.0,
    max=10.0,
    step=0.5,
    description='Cutoff Radius (r_cut):',
    continuous_update=False,
    style={'description_width': 'initial'}
)

n_max_widget = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=1,
    description='n_max:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

l_max_widget = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=1,
    description='l_max:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

similarity_threshold_widget = widgets.FloatSlider(
    value=0.99999,
    min=0.9,
    max=1.0,
    step=0.00001,
    description='Similarity Threshold:',
    continuous_update=False,
    style={'description_width': 'initial'}
)

# Display widgets
display(r_cut_widget, n_max_widget, l_max_widget, similarity_threshold_widget)

output_step_5 = widgets.Output()

def run_step_5(change):
    from pyiron_workflow_atomistics.gb.segregation import calculate_substitutional_segregation_GB, get_unique_sites_SOAP
    from pyiron_workflow_lammps.engine import LammpsEngine

    with output_step_5:
        output_step_5.clear_output()

        # Check if Step 4 outputs exist
        if not hasattr(wf.pure_gb_study.outputs, "pure_grain_boundary_structure_vacuum"):
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Error:</strong> Step 4 outputs are missing. Please run Step 4 first.
            </div>
            """))
            return

        if not hasattr(wf, "soln_energy"):
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Error:</strong> Step 2 outputs are missing. Please run Step 2 first.
            </div>
            """))
            return

        # Collect parameter info for collapsible
        gb_analysis = wf.pure_gb_study.outputs.gb_plane_analysis_dict.value
        total_gb_sites = len(gb_analysis['extended_sel_indices'])

        param_lines = [
            f"Host element: Fe (BCC)",
            f"Solute element: Al",
            f"Total GB sites identified (Step 4): {total_gb_sites}",
            f"SOAP cutoff radius (r_cut): {r_cut_widget.value} Å",
            f"SOAP n_max: {n_max_widget.value}",
            f"SOAP l_max: {l_max_widget.value}",
            f"Similarity threshold: {similarity_threshold_widget.value}",
            f"PCA variance threshold: 0.999",
            f"Working directory: gb_seg_lammps",
            f"Potential: EAM/FS (Al-Fe)",
        ]

        display(HTML(f"""
        <details>
        <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px;">
            Analysis Parameters (click to expand)
        </summary>
        <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
            <ul style="margin:0;">
                {"".join(f"<li>{line}</li>" for line in param_lines)}
            </ul>
        </div>
        </details>
        """))

        display(HTML("""
        <div style="background:#e8f4f8; padding:10px; border-radius:5px; margin:10px 0; border-left:4px solid #3498db;">
            Running Step 5: Segregation Study. This may take several minutes...
        </div>
        <div style="font-weight:bold; padding:6px 0 4px 0; color:#2c3e50;">Phase 1: Identifying unique sites via SOAP descriptors...</div>
        """))

        # Remove existing nodes
        if "site_duplicate_df" in wf.children:
            wf.remove_child("site_duplicate_df")
        if "gb_seg_calcs" in wf.children:
            wf.remove_child("gb_seg_calcs")

        # Identify unique sites
        wf.site_duplicate_df = get_unique_sites_SOAP(
            structure=wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum,
            # defect_sites=wf.pure_gb_study.outputs.gb_plane_analysis_dict["extended_sel_indices"],
            defect_sites=wf.pure_gb_study.outputs.gb_plane_analysis_dict.value["extended_sel_indices"],
            r_cut=r_cut_widget.value,
            n_max=n_max_widget.value,
            l_max=l_max_widget.value,
            n_jobs=-1,
            periodic=True,
            pca_variance_threshold=0.999,
            similarity_threshold=similarity_threshold_widget.value
        )

        wf.run()

        # Site deduplication summary
        if hasattr(wf.site_duplicate_df.outputs, "unique_sites_list"):
            unique_sites = wf.site_duplicate_df.outputs.unique_sites_list.value
            reduction_pct = (1 - len(unique_sites) / total_gb_sites) * 100

            display(HTML(f"""
            <details open>
            <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                Site Deduplication Results (click to expand)
            </summary>
            <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
                <ul style="margin:0;">
                    <li>Total GB sites: <strong>{total_gb_sites}</strong></li>
                    <li>Unique sites after SOAP deduplication: <strong>{len(unique_sites)}</strong></li>
                    <li>Reduction: <strong>{reduction_pct:.1f}%</strong></li>
                </ul>
            </div>
            </details>
            """))
        else:
            unique_sites = []
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Warning:</strong> Could not retrieve unique sites list.
            </div>
            """))

        display(HTML(f"""
        <div style="font-weight:bold; padding:6px 0 4px 0; color:#2c3e50; margin-top:10px;">
            Phase 2: Calculating segregation energies for {len(unique_sites)} unique sites...
        </div>
        """))

        # Create FRESH engine
        inp_seg = CalcInputMinimize()
        inp_seg.relax_cell = False
        Engine_segregation = LammpsEngine(EngineInput=inp_seg)
        Engine_segregation.working_directory = "segregation_study"
        Engine_segregation.lammps_log_filepath = "minimize.log"
        Engine_segregation.command = "lmp -in in.lmp -log minimize.log"
        Engine_segregation.input_script_pair_style = "eam/fs"
        Engine_segregation.path_to_model = potential_path
        Engine_segregation.potential_elements = ["Al", "Fe"]

        # Calculate segregation energies
        wf.gb_seg_calcs = calculate_substitutional_segregation_GB(
            structure=wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum,
            defect_sites=wf.site_duplicate_df.outputs.unique_sites_list,
            element="Al",
            structure_basename="pureGB_Fe_seg",
            parent_dir="gb_seg_lammps",
            calculation_engine=Engine_segregation,
            unique_sites_df=wf.site_duplicate_df.outputs.df,
            df_filename="seg_calcs_df.pkl",
        )

        wf.run()

        # Results summary
        if hasattr(wf.gb_seg_calcs.outputs, "gb_seg_calcs_df"):
            seg_df = wf.gb_seg_calcs.outputs.gb_seg_calcs_df.value
            result_lines = []

            result_lines.append(f"<strong>Calculation Summary</strong>")
            result_lines.append(f"Total sites calculated: {len(seg_df)}")

            # Compute Eseg if energies are available
            if "calc_output" in seg_df.columns:
                try:
                    seg_df = seg_df.copy()
                    seg_df["energy"] = seg_df["calc_output"].apply(lambda x: x.final_energy)
                    pure_gb_energy = wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum_energy.value
                    soln_e = wf.soln_energy.outputs.solution_energy.value
                    seg_df["Eseg"] = seg_df["energy"] - pure_gb_energy - soln_e

                    result_lines.append(f"<strong>Segregation Energies (E_seg)</strong>")
                    result_lines.append(f"Pure GB total energy: {pure_gb_energy:.6f} eV")
                    result_lines.append(f"Solution energy (bulk): {soln_e:.6f} eV")
                    result_lines.append(f"Min E_seg: {seg_df['Eseg'].min():.4f} eV")
                    result_lines.append(f"Max E_seg: {seg_df['Eseg'].max():.4f} eV")
                    result_lines.append(f"Mean E_seg: {seg_df['Eseg'].mean():.4f} eV")

                    favorable = (seg_df['Eseg'] < 0).sum()
                    result_lines.append(f"Sites with favorable segregation (E_seg < 0): {favorable} / {len(seg_df)}")

                    # Table of results
                    display_df = seg_df[["Eseg", "energy"]].copy()
                    display_df.index.name = "Site index"
                    display_df.columns = ["E_seg (eV)", "Total energy (eV)"]

                    display(HTML(f"""
                    <details open>
                    <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                        Step 5 Results (click to expand)
                    </summary>
                    <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
                        <ul style="margin:0 0 10px 0;">
                            {"".join(f"<li>{line}</li>" for line in result_lines)}
                        </ul>
                    </div>
                    </details>
                    """))

                    display(HTML(f"""
                    <details>
                    <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                        Segregation Energies Table (click to expand)
                    </summary>
                    <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db; overflow-x:auto;">
                        {display_df.sort_values("E_seg (eV)").to_html(border=0, classes='table', float_format=lambda x: f"{x:.4f}")}
                    </div>
                    </details>
                    """))

                except Exception as e:
                    display(HTML(f"""
                    <details open>
                    <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                        Step 5 Results (click to expand)
                    </summary>
                    <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
                        <ul style="margin:0;">
                            {"".join(f"<li>{line}</li>" for line in result_lines)}
                        </ul>
                        <p style="color:#c0392b; margin-top:8px;">Note: Could not compute E_seg automatically: {e}</p>
                    </div>
                    </details>
                    """))
            else:
                display(HTML(f"""
                <details open>
                <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                    Step 5 Results (click to expand)
                </summary>
                <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
                    <ul style="margin:0;">
                        {"".join(f"<li>{line}</li>" for line in result_lines)}
                    </ul>
                </div>
                </details>
                """))
        else:
            display(HTML("""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Warning:</strong> No segregation output available. Check that all previous steps completed successfully.
            </div>
            """))
            return

        display(HTML("""
        <div style="background:#e6f4ea; padding:10px; border-radius:5px; margin-top:10px; border-left:4px solid #2e7d32;">
            Step 5 completed successfully. Segregation energies are stored in <code>wf.gb_seg_calcs</code>.
        </div>
        """))

# Add a button to trigger Step 5
run_button_step_5 = widgets.Button(description="Run Step 5")
run_button_step_5.on_click(run_step_5)

display(run_button_step_5, output_step_5)

In [ ]:
# # # --- Workflow Execution ---
# # display(HTML("""
# # <div class="section">
# #     <h2>Workflow Execution</h2>
# #     <ul>
# #         <li>Executes all nodes that can be computed with available inputs</li>
# #     </ul>
# #     <p><strong>Expected Output:</strong></p>
# #     <ul>
# #         <li>Progress bars for GB structure generation</li>
# #         <li>LAMMPS calculation logs</li>
# #         <li>Status messages for each workflow stage</li>
# #     </ul>
# #     <p><strong>Execution Time:</strong> This may take several minutes to hours depending on:</p>
# #     <ul>
# #         <li>Number of GB structures found</li>
# #         <li>Number of unique segregation sites</li>
# #         <li>Computational resources available</li>
# #         <li>Type of engine used (eam potential/ml potential/)</li>
# #     </ul>
# # </div>
# # """))

# # # wf.run()

# # import logging
# # import io
# # from contextlib import redirect_stdout, redirect_stderr

# # # quiet global and specific noisy loggers
# # logging.getLogger().setLevel(logging.WARNING)
# # logging.getLogger("pyiron_workflow").setLevel(logging.WARNING)
# # logging.getLogger("pyiron_workflow_atomistics").setLevel(logging.WARNING)

# # # run and capture textual output in memory; keep display outputs (figures) intact
# # buf_out = io.StringIO()
# # buf_err = io.StringIO()
# # try:
# #     with redirect_stdout(buf_out), redirect_stderr(buf_err):
# #         wf.run()
# # except Exception:
# #     # show captured text for debugging
# #     print("Workflow failed; stdout:")
# #     print(buf_out.getvalue())
# #     print("Workflow failed; stderr:")
# #     print(buf_err.getvalue())
# #     raise

# # --- Workflow Execution ---
# display(HTML("""
# <div class="section">
#     <h2>Workflow Execution</h2>
#     <ul>
#         <li>Executes all nodes that can be computed with available inputs</li>
#     </ul>
#     <p><strong>Expected Output:</strong></p>
#     <ul>
#         <li>Progress bars for GB structure generation</li>
#         <li>LAMMPS calculation logs</li>
#         <li>Status messages for each workflow stage</li>
#     </ul>
#     <p><strong>Execution Time:</strong> This may take several minutes to hours depending on:</p>
#     <ul>
#         <li>Number of GB structures found</li>
#         <li>Number of unique segregation sites</li>
#         <li>Computational resources available</li>
#         <li>Type of engine used (eam potential/ml potential/)</li>
#     </ul>
# </div>
# """))

# import logging
# import io
# from contextlib import redirect_stdout, redirect_stderr

# # Quiet global and specific noisy loggers
# logging.getLogger().setLevel(logging.WARNING)
# logging.getLogger("pyiron_workflow").setLevel(logging.WARNING)
# logging.getLogger("pyiron_workflow_atomistics").setLevel(logging.WARNING)

# # Run and capture textual output in memory; keep display outputs (figures) intact
# buf_out = io.StringIO()
# buf_err = io.StringIO()
# try:
#     with redirect_stdout(buf_out), redirect_stderr(buf_err):
#         print("Starting workflow execution...")
#         wf.run()
#         print("Workflow execution completed.")

#         # Debugging: Check and display outputs for each node
#         if hasattr(wf, "outputs"):
#             print("Workflow outputs:")
#             for node_name, node in wf.outputs.items():
#                 print(f"Node: {node_name}")
#                 if hasattr(node, "plot"):
#                     print(f"Attempting to plot output for node: {node_name}")
#                     plot_output = node.plot()
#                     if plot_output:
#                         display(plot_output)
#                     else:
#                         print(f"Plot method exists for {node_name} but returned no output.")
#                 else:
#                     print(f"No plot method available for node: {node_name}")
#         else:
#             print("No outputs available in the workflow.")
# except Exception as e:
#     # Show captured text for debugging
#     print("Workflow failed; stdout:")
#     print(buf_out.getvalue())
#     print("Workflow failed; stderr:")
#     print(buf_err.getvalue())
#     print(f"Exception: {e}")
#     raise

In [ ]:
# --- Results Analysis ---
display(HTML("""
<div class="section">
    <h2>Results Analysis</h2>
    <p>Now that the workflow has completed, let's analyze the segregation results.</p>
    <h3>Data Processing</h3>
    <p>We will:</p>
    <ul>
        <li><strong>Extract energies</strong> from the calculation outputs</li>
        <li><strong>Calculate segregation energies</strong> for each site using:
            <br><span style="font-family:monospace;">E_seg = (E_GB+Al - E_GB_pure) - E_solution</span>
        </li>
        <li><strong>Compute the distance</strong> of each site from the GB plane</li>
        <li><strong>Organize results</strong> in a DataFrame and offer CSV download</li>
    </ul>
</div>
"""))

output_results = widgets.Output()

def run_results_analysis(change):
    with output_results:
        output_results.clear_output()

        # Check prerequisites
        missing = []
        if not hasattr(wf, "gb_seg_calcs") or not hasattr(wf.gb_seg_calcs.outputs, "gb_seg_calcs_df"):
            missing.append("Step 5 (gb_seg_calcs)")
        if not hasattr(wf, "pure_gb_study") or not hasattr(wf.pure_gb_study.outputs, "pure_grain_boundary_structure_vacuum_energy"):
            missing.append("Step 4 (pure_gb_study)")
        if not hasattr(wf, "soln_energy") or not hasattr(wf.soln_energy.outputs, "solution_energy"):
            missing.append("Step 2 (soln_energy)")

        if missing:
            display(HTML(f"""
            <div style="background:#fff3cd; padding:10px; border-radius:5px; border-left:4px solid #f0a500;">
                <strong>Error:</strong> The following steps are missing or incomplete: {", ".join(missing)}.
                Please run all previous steps first.
            </div>
            """))
            return

        display(HTML("""
        <div style="background:#e8f4f8; padding:10px; border-radius:5px; margin:10px 0; border-left:4px solid #3498db;">
            Processing segregation results...
        </div>
        """))

        try:
            df = wf.gb_seg_calcs.outputs.gb_seg_calcs_df.value.copy()
            df["energy"] = df["calc_output"].apply(lambda x: x.final_energy)

            pure_gb_energy = wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum_energy.value
            soln_e = wf.soln_energy.outputs.solution_energy.value

            df["Eseg"] = df["energy"] - pure_gb_energy - soln_e

            gb_pos = wf.pure_gb_study.outputs.gb_plane_analysis_dict.value["gb_cart"]

            def get_site_gb_distance(row):
                struct = row["structure"]
                rep_idx = row["rep"]
                dist = np.round(np.abs(struct.positions[rep_idx][2] - gb_pos), 1)
                return dist

            df["dist_GB"] = df.apply(get_site_gb_distance, axis=1)
            df_sorted = df.sort_values(by="dist_GB").reset_index(drop=True)

            # Summary stats
            result_lines = [
                f"<strong>Summary Statistics</strong>",
                f"Total unique sites analyzed: {len(df_sorted)}",
                f"Pure GB total energy: {pure_gb_energy:.6f} eV",
                f"Solution energy (bulk): {soln_e:.6f} eV",
                f"GB plane position (Cartesian): {gb_pos:.4f} Å",
                f"<strong>Segregation Energy Statistics</strong>",
                f"Min E_seg: {df_sorted['Eseg'].min():.4f} eV (at dist_GB = {df_sorted.loc[df_sorted['Eseg'].idxmin(), 'dist_GB']:.1f} Å)",
                f"Max E_seg: {df_sorted['Eseg'].max():.4f} eV",
                f"Mean E_seg: {df_sorted['Eseg'].mean():.4f} eV",
                f"Sites with favorable segregation (E_seg < 0): {(df_sorted['Eseg'] < 0).sum()} / {len(df_sorted)}",
            ]

            display(HTML(f"""
            <details open>
            <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                Analysis Summary (click to expand)
            </summary>
            <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db;">
                <ul style="margin:0;">
                    {"".join(f"<li>{line}</li>" for line in result_lines)}
                </ul>
            </div>
            </details>
            """))

            # Build export DataFrame — ALL rows
            export_cols = ["dist_GB", "Eseg", "energy"]
            if "rep" in df_sorted.columns:
                export_cols.insert(0, "rep")
            export_df = df_sorted[export_cols].copy()  # full dataset, no head()
            export_df.index.name = "index"
            col_rename = {"dist_GB": "dist_GB (Å)", "Eseg": "E_seg (eV)", "energy": "Total energy (eV)"}
            if "rep" in export_df.columns:
                col_rename["rep"] = "Site index"
            export_df = export_df.rename(columns=col_rename)

            # CSV download via base64 — encode full export_df
            import base64
            csv_str = export_df.to_csv(index=True)  # all rows
            b64 = base64.b64encode(csv_str.encode()).decode()
            href = f'data:text/csv;base64,{b64}'

            display(HTML(f"""
            <div style="margin-top:15px; padding:12px; background:#f1f8ff; border-radius:5px; border:1px solid #d1e5f9;">
                <strong>Download Results:</strong>
                <a href="{href}" download="segregation_results.csv"
                   style="margin-left:10px; padding:6px 14px; background:#3498db; color:white;
                          border-radius:4px; text-decoration:none; font-weight:bold;">
                    ⬇ Download CSV
                </a>
                <span style="margin-left:10px; color:#7f8c8d; font-size:0.9em;">
                    segregation_results.csv — {len(export_df)} rows, sorted by distance from GB
                </span>
            </div>
            """))

            # Preview only — first 10 rows for display, does NOT affect the CSV above
            n_remaining = len(export_df) - 10
            remaining_note = (
                f"<p style='color:#7f8c8d; font-style:italic; margin-top:8px;'>"
                f"... and {n_remaining} more rows in the downloaded CSV.</p>"
                if n_remaining > 0 else ""
            )

            display(HTML(f"""
            <details>
            <summary style="cursor:pointer; font-weight:bold; padding:10px; background:#f1f8ff; border-radius:5px; margin-top:10px;">
                Preview — first 10 of {len(export_df)} rows sorted by dist_GB (click to expand)
            </summary>
            <div style="padding:10px; background:#f8f9fa; border-radius:0 0 5px 5px; border-left:4px solid #3498db; overflow-x:auto;">
                {export_df.head(10).to_html(border=0, classes='table', float_format=lambda x: f"{x:.4f}")}
                {remaining_note}
            </div>
            </details>
            """))

            display(HTML("""
            <div style="background:#e6f4ea; padding:10px; border-radius:5px; margin-top:10px; border-left:4px solid #2e7d32;">
                Results Analysis completed. Use the Download CSV button above to save the full results table.
            </div>
            """))

        except Exception as e:
            display(HTML(f"""
            <div style="background:#fdecea; padding:10px; border-radius:5px; border-left:4px solid #c0392b;">
                <strong>Error during analysis:</strong> {e}
            </div>
            """))

run_button_results = widgets.Button(description="Run Results Analysis")
run_button_results.on_click(run_results_analysis)

display(run_button_results, output_results)

In [ ]:
# df = wf.gb_seg_calcs.outputs.gb_seg_calcs_df.value.copy()
# df["energy"] = df.calc_output.apply(lambda x: x.final_energy)
# df["Eseg"] = (
#     df.energy
#     - wf.pure_gb_study.outputs.pure_grain_boundary_structure_vacuum_energy.value
#     - wf.soln_energy.outputs.solution_energy.value
# )

# import numpy as np

# gb_pos = wf.pure_gb_study.outputs.gb_plane_analysis_dict.value["gb_cart"]

# def get_site_gb_distance(row):
#     struct = row["structure"]
#     rep_idx = row["rep"]
#     dist = np.round(np.abs(struct.positions[rep_idx][2] - gb_pos), 1)
#     return dist

# df["dist_GB"] = df.apply(get_site_gb_distance, axis=1)

# # Show the first 10 rows in a styled section
# display(HTML("""
# <div class="section">
#     <h2>Segregation Results Table</h2>
#     <p>Below are the first 10 segregation results, sorted by distance from the grain boundary:</p>
# </div>
# """))
# display(HTML(df.sort_values(by="dist_GB").head(10).to_html(classes='table table-striped', border=0)))

In [ ]:
os.chdir("..")
import shutil
shutil.rmtree("calculations", ignore_errors=True)